# Week 01 Lab — TRACE/01

첫 호출을 `실행 → 기록 → 비교 → 개선` 가능한 실험으로 바꾼다. 완료 기준은 같은 입력의 trace 2건, 비교표 1개, 다음 version 가설 2문장이다.

## 0. 서비스 실행

저장소 루트의 별도 터미널에서 아래 명령을 실행한 뒤 이 notebook을 진행한다.

```bash
uv python install 3.11.14
uv venv --python 3.11.14 --allow-existing .venv
source .venv/bin/activate
uv pip install -r requirements.txt
uvicorn app.main:app --app-dir week01/lab --reload
```

In [ ]:
from __future__ import annotations

import os
import platform
import sys

import httpx
import pandas as pd

BASE_URL = os.getenv("TRACE01_BASE_URL", "http://127.0.0.1:8000")
assert sys.version_info[:2] == (3, 11), sys.version
print({"python": platform.python_version(), "service": BASE_URL})

## 1. Preflight

모델을 호출하기 전에 서비스 계약부터 확인한다. 여기서 실패하면 prompt가 아니라 실행 경로 문제다.

In [ ]:
health = httpx.get(f"{BASE_URL}/health", timeout=5.0)
health.raise_for_status()
config = httpx.get(f"{BASE_URL}/api/v1/config", timeout=5.0)
config.raise_for_status()
{"health": health.json(), "ollama": config.json()["ollama"]}

## 2. Baseline 두 번 실행

입력·task·provider·temperature를 고정한다. demo provider는 네트워크나 API key 없이 모두가 같은 루프를 완주하도록 결정론적으로 동작한다.

In [ ]:
def generate(*, text: str, task: str, provider: str = "demo", temperature: float = 0.2) -> dict:
    response = httpx.post(
        f"{BASE_URL}/api/v1/generate",
        json={
            "text": text,
            "task": task,
            "provider": provider,
            "temperature": temperature,
        },
        timeout=90.0,
    )
    response.raise_for_status()
    return response.json()


INPUT = (
    "배송은 빨랐지만 추천 결과가 매번 달라서 비교하기 어려웠습니다. "
    "어떤 프롬프트와 모델을 썼는지 기록되면 팀이 더 빠르게 개선할 수 있을 것 같아요."
)

baseline_runs = [generate(text=INPUT, task="extract") for _ in range(2)]
baseline_runs[0]

## 3. 응답과 trace를 함께 비교

좋은 응답만 보지 않는다. prompt version, model, status, latency, token 추정치, 동일 입력 fingerprint가 한 행에 있어야 한다.

In [ ]:
comparison = pd.DataFrame(
    [
        {
            "run": index,
            "output": run["output"],
            **run["trace"],
        }
        for index, run in enumerate(baseline_runs, start=1)
    ]
)
comparison[
    [
        "run",
        "trace_id",
        "prompt_version",
        "provider",
        "model",
        "temperature",
        "thinking_requested",
        "output_token_limit",
        "status",
        "latency_ms",
        "output_tokens_est",
        "content_fingerprint",
    ]
]

## 4. 같은 실행을 세 가지 관측 형태로 보기

개별 trace, 수업용 집계, Prometheus text가 같은 요청을 서로 다른 운영 관점으로 보여준다. 기본 trace에는 원문 대신 fingerprint만 저장한다. fingerprint는 익명화가 아니므로 개인정보·기밀 대신 합성 입력을 사용한다.

In [ ]:
recent_traces = httpx.get(f"{BASE_URL}/api/v1/traces", params={"limit": 5}).json()
stats = httpx.get(f"{BASE_URL}/api/v1/stats").json()
metrics = httpx.get(f"{BASE_URL}/metrics").text

print("STATS", stats)
print("\nMETRICS\n", metrics)
pd.DataFrame(recent_traces)

## 5. 선택: Ollama로 provider 하나만 변경

수업 표준 모델은 Thinking 전용 태그가 아닌 `qwen3:4b-instruct`다. `ollama pull qwen3:4b-instruct`와 `cp week01/lab/.env.example week01/lab/.env`를 준비하고 TRACE/01을 `.env` 작업으로 다시 실행한다. 같은 입력·task·temperature를 유지한 채 provider만 바꾸면 모델 적재 시간과 생성 시간을 함께 비교할 수 있다. `qwen3:4b`와 `--hidethinking`은 속도 개선용 설정으로 사용하지 않는다.

In [ ]:
try:
    ollama_run = generate(text=INPUT, task="extract", provider="ollama", temperature=0.2)
    print(ollama_run["output"])
    display(
        pd.DataFrame(
            [ollama_run["trace"]],
            columns=[
                "trace_id",
                "model",
                "thinking_requested",
                "output_token_limit",
                "model_load_ms",
                "model_generation_ms",
                "model_output_tokens",
                "finish_reason",
                "latency_ms",
            ],
        )
    )
except httpx.HTTPStatusError as exc:
    print("Ollama 선택 경로를 건너뜁니다:", exc.response.json().get("detail", exc))
except httpx.RequestError as exc:
    print("TRACE/01 연결을 확인하세요:", exc)

## Exit ticket

1. trace가 없었다면 무엇을 비교하지 못했는가?
2. 다음 version에서 바꿀 변수 하나는 무엇인가?
3. 개선 여부를 어떤 지표와 실패 사례로 판단할 것인가?